# 06 — Audio/Video Feature Analysis

Check alignment, missing modalities, feature dimensions, and feature-cache manifests before fusion.

> **Project:** Multimodal Deepfake Detection using LAV-DF (audio + video). Run cells in order. Confirm dataset paths and split definitions before training. Never fit preprocessing or tune thresholds on the test split.

In [ ]:
from pathlib import Path
import pandas as pd

MANIFESTS = {
    "video": Path("../../data/processed/video_manifest.csv"),
    "audio": Path("../../data/processed/audio_manifest.csv"),
}
for modality, path in MANIFESTS.items():
    print(f"{modality}: {path.resolve()} | exists={path.exists()}")

## Alignment contract
Every video feature and audio feature must map to the same stable sample ID and split. Never join by row order or filename substring alone. Check duplicates and unmatched IDs before building a multimodal dataset.

In [ ]:
if all(p.exists() for p in MANIFESTS.values()):
    video_df = pd.read_csv(MANIFESTS["video"])
    audio_df = pd.read_csv(MANIFESTS["audio"])
    print("Video rows:", len(video_df), "Audio rows:", len(audio_df))
    if "sample_id" in video_df.columns and "sample_id" in audio_df.columns:
        print("Video duplicate IDs:", video_df["sample_id"].duplicated().sum())
        print("Audio duplicate IDs:", audio_df["sample_id"].duplicated().sum())
        print("Video without audio:", len(set(video_df.sample_id) - set(audio_df.sample_id)))
        print("Audio without video:", len(set(audio_df.sample_id) - set(video_df.sample_id)))
else:
    print("Manifests not ready. Generate them in notebooks 03 and 04 first.")

### Leakage checks
Ensure all segments or derived samples from a source video remain in the same split. Fit normalization statistics using training data only and reuse them unchanged for development and test.